## Trained model and optimal hyperparameters

We have provided trained model and hyperparameters of SphereNet on QM9 and MD17 [here](https://github.com/divelab/DIG_storage/tree/main/3dgraph).

## Example of 3D Graph

Here we provide the example code for SphereNet on QM93D and MD17 datasets. You can easily replace SphereNet with SchNet and DimeNetPP by changing model name and model parameters.

In [6]:
import torch
import sys
sys.path.insert(0,'..')
sys.path.insert(0,'../..')
from dig.threedgraph.dataset import QM93D
from dig.threedgraph.dataset import MD17
from dig.threedgraph.method import SphereNet, SchNet, DimeNetPP
from dig.threedgraph.method import run
from dig.threedgraph.evaluation import ThreeDEvaluator

import numpy as np
import os.path as osp

In [7]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda', index=0)

### Example code for QM93D data
***Note***: '3D' means that the data includes positional information for atoms.

We trained a separate model for each target except for _gap_, which was predicted by taking _lumo-homo_. You can use default hyperparameters to get comparable results, we also tuned hyperparameters like lr, lr_decay_factor, lr_decay_step_size, cutoff, num_spherical, num_radial, basis_emb_size_dist, basis_emb_size_angle, basis_emb_size_torsion to achieve better performance. The values/search space for hyperparameters are listed in the Appendix of our paper.

The default hyperparameters for QM93D are:  
    &ensp; energy_and_force=False, cutoff=5.0, num_layers=4, hidden_channels=128, out_channels=1, int_emb_size=64,  
    &ensp; basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256,  
    &ensp; num_spherical=3, num_radial=6, envelope_exponent=5,  
    &ensp; num_before_skip=1, num_after_skip=2, num_output_layers=3,  
    &ensp; epochs=1000, batch_size=32, vt_batch_size=32, lr=0.0005, lr_decay_factor=0.5, lr_decay_step_size=100.




#### Loading dataset

In [8]:
dataset = QM93D(root='dataset/')
target = 'U0'
dataset.data.y = dataset.data[target]

split_idx = dataset.get_idx_split(len(dataset.data.y), train_size=110000, valid_size=10000, seed=42)

train_dataset, valid_dataset, test_dataset = dataset[split_idx['train']], dataset[split_idx['valid']], dataset[split_idx['test']]
print('train, validaion, test:', len(train_dataset), len(valid_dataset), len(test_dataset))

train, validaion, test: 110000 10000 10831


#### Loading model, loss and evaluation function

The evaluation metric is mean absolute error (MAE).

In [9]:
model = SphereNet(cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3, use_node_features=True, dropout_rate=0.2
        )
loss_func = torch.nn.L1Loss()
evaluation = ThreeDEvaluator()

#### Training

In [10]:
run3d = run()
run3d.run(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.0005, lr_decay_factor=0.5, lr_decay_step_size=15)

#Params: 1890118

=====Epoch 1

Training...


100%|██████████| 3438/3438 [09:03<00:00,  6.32it/s]



Evaluating...



100%|██████████| 313/313 [00:08<00:00, 36.42it/s]



Testing...


...



Testing...



100%|██████████| 339/339 [00:07<00:00, 43.93it/s]


{'Train': 0.6300246347654275, 'Validation': 0.2120869755744934, 'Test': 0.2130744755268097}

=====Epoch 5

Training...



 14%|█▍        | 498/3438 [01:16<07:32,  6.50it/s]


KeyboardInterrupt: 

In [11]:
model = SphereNet(cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3, use_node_features=True, dropout_rate=0.0
        )
loss_func = torch.nn.L1Loss()
evaluation = ThreeDEvaluator()
run3d = run()
run3d.run(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.0005, lr_decay_factor=0.5, lr_decay_step_size=15)

#Params: 1890118

=====Epoch 1

Training...


100%|██████████| 3438/3438 [08:43<00:00,  6.56it/s]



Evaluating...



100%|██████████| 313/313 [00:07<00:00, 43.70it/s]



Testing...


...



Evaluating...



100%|██████████| 313/313 [00:06<00:00, 44.91it/s]



Testing...



100%|██████████| 339/339 [00:07<00:00, 44.03it/s]


{'Train': 0.05113282303015391, 'Validation': 0.03271997720003128, 'Test': 0.032534874975681305}
Best validation Loss so far: 0.03271997720003128
Test Loss when got best validation result: 0.032534874975681305


In [ ]:
from dig.threedgraph.dataset import QM93D
from dig.threedgraph.method import SphereNet
from dig.threedgraph.evaluation import ThreeDEvaluator
import argparse
import os
import torch
from torch_geometric.data import DataLoader
from tqdm import tqdm


parser = argparse.ArgumentParser(description='QM9 SphereNet')
parser.add_argument('--device', type=int, default=0)

parser.add_argument('--cutoff', type=float, default=5.0)
parser.add_argument('--num_layers', type=int, default=4)
parser.add_argument('--hidden_channels', type=int, default=128)
parser.add_argument('--out_channels', type=int, default=1)
parser.add_argument('--int_emb_size', type=int, default=64)
parser.add_argument('--basis_emb_size_dist', type=int, default=8)
parser.add_argument('--basis_emb_size_angle', type=int, default=8)
parser.add_argument('--basis_emb_size_torsion', type=int, default=8)
parser.add_argument('--out_emb_channels', type=int, default=256)
parser.add_argument('--num_spherical', type=int, default=3)
parser.add_argument('--num_radial', type=int, default=6)

parser.add_argument('--epochs', type=int, default=1000)
parser.add_argument('--batch_size', type=int, default=32)
parser.add_argument('--vt_batch_size', type=int, default=32)
parser.add_argument('--lr', type=float, default=0.0005)
parser.add_argument('--lr_decay_factor', type=float, default=0.5)
parser.add_argument('--lr_decay_step_size', type=int, default=100)

args = parser.parse_args()
print(args)


In [ ]:

device = f'cuda:{args.device}' if torch.cuda.is_available() else 'cpu'
device = torch.device(device)
print('device',device)

save_dir='qm9' 
vt_batch_size = 32

dataset = QM93D(root='dataset/', cutoff=args.cutoff)
split_idx = dataset.get_idx_split(len(dataset.data.y), train_size=110000, valid_size=10000, seed=42)

model = SphereNet(energy_and_force=False, cutoff=args.cutoff, num_layers=args.num_layers, 
            hidden_channels=args.hidden_channels, out_channels=args.out_channels, int_emb_size=args.int_emb_size, 
            basis_emb_size_dist=args.basis_emb_size_dist, basis_emb_size_angle=args.basis_emb_size_angle, basis_emb_size_torsion=args.basis_emb_size_torsion, out_emb_channels=args.out_emb_channels, 
            num_spherical=args.num_spherical, num_radial=args.num_radial, envelope_exponent=5, 
            num_before_skip=1, num_after_skip=2, num_output_layers=3 
            )
model = model.to(device)

evaluation = ThreeDEvaluator()


for target in ['mu','alpha','homo','lumo','zpve','U0','U','H','G','Cv']:
    dataset.data.y = dataset.data[target]
    train_dataset, valid_dataset, test_dataset = dataset[split_idx['train']], dataset[split_idx['valid']], dataset[split_idx['test']]
    print('train, validaion, test:', len(train_dataset), len(valid_dataset), len(test_dataset))
    test_loader = DataLoader(test_dataset, vt_batch_size, shuffle=False)
    
    checkpoint = torch.load(os.path.join(save_dir, target+'_checkpoint.pt'))
    model.load_state_dict(checkpoint['model_state_dict'])

    model.eval()

    preds = torch.Tensor([]).to(device)
    targets = torch.Tensor([]).to(device)

    for step, batch_data in enumerate(tqdm(test_loader)):
        batch_data = batch_data.to(device)
        with torch.no_grad():
            out = model(batch_data)
        preds = torch.cat([preds, out.detach_()], dim=0)
        targets = torch.cat([targets, batch_data.y.unsqueeze(1)], dim=0)

    mae = torch.mean(torch.abs(preds - targets)).cpu().item()
    print('mae:',target, mae)

In [ ]:
from rdkit import Chem

mol = Chem.MolFromSmiles('CCCO')
mol
AllChem.EmbedMultipleConfs(mol, numConfs=2, randomSeed=2222, numThreads=0)

root = 'dataset/qm9/raw/'
raw_file_names = 'qm9_eV.npz'
data = np.load(osp.join(root, raw_file_names))
R = data['R']
Z = data['Z']
N = data['N']
print(len(R))
print(len(Z))
print(len(N))
split = np.cumsum(N)
R_qm9 = np.split(R, split)
Z_qm9 = np.split(Z, split)
print(R_qm9)
print(Z_qm9)

for i in data:
    print(i)
print('-----'*10)    
for i in data['Cv']:
    print(i)
    break
    
for name in ['mu', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve','U0', 'U', 'H', 'G', 'Cv']:
    print(name)
    print(dataset.data[name][0])